# Sorting & Searching — Coding Interview Prep

Binary search, heap operations, and sorting algorithms.

## 1. Binary Search Variants

In [ ]:
# Standard binary search
def binary_search(nums, target):
    lo, hi = 0, len(nums) - 1
    while lo <= hi:
        mid = lo + (hi - lo) // 2  # avoids overflow
        if nums[mid] == target: return mid
        elif nums[mid] < target: lo = mid + 1
        else: hi = mid - 1
    return -1

# First/last occurrence (leftmost)
def search_range(nums, target):
    def find_left(nums, target):
        lo, hi, idx = 0, len(nums) - 1, -1
        while lo <= hi:
            mid = (lo + hi) // 2
            if nums[mid] == target: idx = mid; hi = mid - 1
            elif nums[mid] < target: lo = mid + 1
            else: hi = mid - 1
        return idx

    def find_right(nums, target):
        lo, hi, idx = 0, len(nums) - 1, -1
        while lo <= hi:
            mid = (lo + hi) // 2
            if nums[mid] == target: idx = mid; lo = mid + 1
            elif nums[mid] < target: lo = mid + 1
            else: hi = mid - 1
        return idx

    return [find_left(nums, target), find_right(nums, target)]

# Search in rotated sorted array
def search_rotated(nums, target):
    lo, hi = 0, len(nums) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if nums[mid] == target: return mid
        if nums[lo] <= nums[mid]:  # left half sorted
            if nums[lo] <= target < nums[mid]: hi = mid - 1
            else: lo = mid + 1
        else:  # right half sorted
            if nums[mid] < target <= nums[hi]: lo = mid + 1
            else: hi = mid - 1
    return -1

# Find minimum in rotated array
def find_min_rotated(nums):
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] > nums[hi]: lo = mid + 1
        else: hi = mid
    return nums[lo]

assert binary_search([1,3,5,7,9], 7) == 3
assert search_range([5,7,7,8,8,10], 8) == [3, 4]
assert search_rotated([4,5,6,7,0,1,2], 0) == 4
assert find_min_rotated([3,4,5,1,2]) == 1
print("Binary search: all tests passed")

## 2. Binary Search on Answer

In [ ]:
import math

# Koko eating bananas: find minimum speed to eat all piles in h hours
def min_eating_speed(piles, h):
    lo, hi = 1, max(piles)
    while lo < hi:
        mid = (lo + hi) // 2
        hours = sum(math.ceil(p / mid) for p in piles)
        if hours <= h: hi = mid
        else: lo = mid + 1
    return lo

assert min_eating_speed([3, 6, 7, 11], 8) == 4

# Capacity to ship packages in D days
def ship_capacity(weights, days):
    lo, hi = max(weights), sum(weights)
    while lo < hi:
        mid = (lo + hi) // 2
        # Greedy: count days needed for capacity=mid
        needed = curr = 1
        for w in weights:
            if curr + w > mid: needed += 1; curr = 0
            curr += w
        if needed <= days: hi = mid
        else: lo = mid + 1
    return lo

assert ship_capacity([1,2,3,4,5,6,7,8,9,10], 5) == 15
print("Binary search on answer: all tests passed")

## 3. Heap Problems

In [ ]:
import heapq

# K largest elements
def k_largest(nums, k):
    return heapq.nlargest(k, nums)  # O(n log k)

# Kth largest (heap approach)
def kth_largest(nums, k):
    min_heap = []
    for n in nums:
        heapq.heappush(min_heap, n)
        if len(min_heap) > k:
            heapq.heappop(min_heap)
    return min_heap[0]

assert kth_largest([3,2,1,5,6,4], 2) == 5
assert kth_largest([3,2,3,1,2,4,5,5,6], 4) == 4

# Merge K sorted lists
def merge_k_sorted(lists):
    heap = []  # (value, list_idx, element_idx)
    for i, lst in enumerate(lists):
        if lst: heapq.heappush(heap, (lst[0], i, 0))

    result = []
    while heap:
        val, i, j = heapq.heappop(heap)
        result.append(val)
        if j + 1 < len(lists[i]):
            heapq.heappush(heap, (lists[i][j+1], i, j+1))
    return result

assert merge_k_sorted([[1,4,5],[1,3,4],[2,6]]) == [1,1,2,3,4,4,5,6]

# Top K frequent elements
from collections import Counter
def top_k_frequent(nums, k):
    return [x for x, _ in Counter(nums).most_common(k)]

assert set(top_k_frequent([1,1,1,2,2,3], 2)) == {1, 2}
print("Heap problems: all tests passed")

## 4. Sorting Algorithms

In [ ]:
import time
import random

def merge_sort(arr):
    if len(arr) <= 1: return arr
    mid = len(arr) // 2
    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])
    return merge(left, right)

def merge(left, right):
    result, i, j = [], 0, 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]: result.append(left[i]); i += 1
        else: result.append(right[j]); j += 1
    return result + left[i:] + right[j:]

def quick_sort(arr, lo=0, hi=None):
    if hi is None: hi = len(arr) - 1
    if lo < hi:
        pivot_idx = partition(arr, lo, hi)
        quick_sort(arr, lo, pivot_idx - 1)
        quick_sort(arr, pivot_idx + 1, hi)
    return arr

def partition(arr, lo, hi):
    pivot = arr[hi]
    i = lo - 1
    for j in range(lo, hi):
        if arr[j] <= pivot:
            i += 1
            arr[i], arr[j] = arr[j], arr[i]
    arr[i+1], arr[hi] = arr[hi], arr[i+1]
    return i + 1

# Verify
data = [3, 6, 8, 10, 1, 2, 1]
assert merge_sort(data[:]) == sorted(data)
assert quick_sort(data[:]) == sorted(data)

# Performance comparison
n = 10000
random_data = [random.randint(0, n) for _ in range(n)]

t0 = time.perf_counter()
merge_sort(random_data[:])
merge_time = time.perf_counter() - t0

t0 = time.perf_counter()
sorted(random_data)
timsort_time = time.perf_counter() - t0

print(f"n={n}: merge_sort={merge_time*1000:.1f}ms, Python Timsort={timsort_time*1000:.1f}ms")

## 5. Quick Select — Kth Element O(n) average

In [ ]:
def quick_select(nums, k):
    """Find kth smallest in average O(n). k is 1-indexed."""
    def select(lo, hi, k):
        if lo == hi: return nums[lo]
        pivot_idx = partition(nums, lo, hi)
        rank = pivot_idx - lo + 1
        if k == rank: return nums[pivot_idx]
        elif k < rank: return select(lo, pivot_idx - 1, k)
        else: return select(pivot_idx + 1, hi, k - rank)
    return select(0, len(nums) - 1, k)

nums = [3, 2, 1, 5, 6, 4]
assert quick_select(nums[:], 2) == 2  # 2nd smallest
assert quick_select(nums[:], 4) == 4  # 4th smallest

# Comparison: heap vs quickselect for kth element
print("Quick select: O(n) avg vs O(n log k) heap vs O(n log n) sort")

## Reference: Complexity Table

| Algorithm | Best | Average | Worst | Space | Stable |
|-----------|------|---------|-------|-------|--------|
| Merge Sort | O(n log n) | O(n log n) | O(n log n) | O(n) | Yes |
| Quick Sort | O(n log n) | O(n log n) | O(n²) | O(log n) | No |
| Heap Sort | O(n log n) | O(n log n) | O(n log n) | O(1) | No |
| Tim Sort (Python) | O(n) | O(n log n) | O(n log n) | O(n) | Yes |
| Binary Search | O(1) | O(log n) | O(log n) | O(1) | — |
| Quick Select | O(n) | O(n) | O(n²) | O(log n) | — |

**Interview tip**: always use Python's `sorted()` / `list.sort()` (Timsort) unless asked to implement. Binary search with `bisect` module saves time.